In [1]:
import os, warnings, logging
import pandas as pd, numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import log_loss

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)



In [2]:
# Paths (adjust if the environment layout changes)
BASE_PATH = "/kaggle/input/lmsys-chatbot-arena"
TRAIN_PATH = os.path.join(BASE_PATH, "train.csv")
TEST_PATH = os.path.join(BASE_PATH, "test.csv")
SUBMISSION_PATH = "submission.csv"

assert os.path.exists(TRAIN_PATH), f"train.csv not found at {TRAIN_PATH}"
assert os.path.exists(TEST_PATH), f"test.csv not found at {TEST_PATH}"
logger.info(f"Found train ({TRAIN_PATH}) and test ({TEST_PATH}) files.")



2026-05-03 04:04:07,094 - INFO - Found train (/kaggle/input/lmsys-chatbot-arena/train.csv) and test (/kaggle/input/lmsys-chatbot-arena/test.csv) files.


In [3]:
# Load data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
logger.info(f"Train rows: {len(train_df)}, Test rows: {len(test_df)}")




2026-05-03 04:04:08,449 - INFO - Train rows: 51729, Test rows: 5748


In [4]:
def _basic_features(df):
    """Create a light set of numeric features."""
    # Character length of each response
    len_a = df["response_a"].astype(str).str.len()
    len_b = df["response_b"].astype(str).str.len()
    # Difference in length
    diff_len = len_a - len_b
    # Ratio (avoid division by zero)
    ratio_len = len_a / (len_b.replace(0, np.nan))
    ratio_len = ratio_len.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    # Optional readability via textstat (may be unavailable)
    try:
        import textstat

        read_a = df["response_a"].astype(str).apply(textstat.flesch_kincaid_grade)
        read_b = df["response_b"].astype(str).apply(textstat.flesch_kincaid_grade)
    except Exception:
        read_a = pd.Series(np.nan, index=df.index)
        read_b = pd.Series(np.nan, index=df.index)
    diff_read = read_a - read_b
    # Assemble
    features = pd.DataFrame(
        {
            "len_a": len_a,
            "len_b": len_b,
            "diff_len": diff_len,
            "ratio_len": ratio_len,
            "read_a": read_a,
            "read_b": read_b,
            "diff_read": diff_read,
        }
    )
    # Fill NaNs with 0 (simple handling)
    return features.fillna(0.0)




In [5]:
# Prepare training features and target
X_train = _basic_features(train_df)
y_train = (
    train_df[["winner_model_a", "winner_model_b", "winner_tie"]]
    .idxmax(axis=1)  # get column name of the winning class
    .map({"winner_model_a": 0, "winner_model_b": 1, "winner_tie": 2})
    .values
)

# Build a pipeline: RobustScaler + multinomial LogisticRegression
model_pipe = make_pipeline(
    RobustScaler(),
    LogisticRegression(
        multi_class="multinomial", max_iter=300, n_jobs=5, class_weight="balanced"
    ),
)
model_pipe.fit(X_train, y_train)
logger.info("Training completed.")



2026-05-03 04:04:09,273 - INFO - Training completed.


In [6]:
# Transform test set and predict probabilities
X_test = _basic_features(test_df)
test_probs = model_pipe.predict_proba(X_test)  # shape (n_test, 3)
# Ensure proper shape/order: columns correspond to [a, b, tie]
assert test_probs.shape[1] == 3, "Model must output three class probabilities."

# Clip and renormalise for numerical stability
eps = 1e-7
test_probs = np.clip(test_probs, eps, 1 - eps)
test_probs = test_probs / test_probs.sum(axis=1, keepdims=True)



In [7]:
# Build submission DataFrame with required column names
submission = pd.DataFrame(
    {
        "id": test_df["id"],
        "winner_model_a": test_probs[:, 0],
        "winner_model_b": test_probs[:, 1],
        "winner_tie": test_probs[:, 2],
    }
)
submission = submission[["id", "winner_model_a", "winner_model_b", "winner_tie"]]
submission.to_csv(SUBMISSION_PATH, index=False)
logger.info(f"✅ Submission saved to {SUBMISSION_PATH}")
print(submission.head())

2026-05-03 04:04:09,313 - INFO - ✅ Submission saved to submission.csv


           id  winner_model_a  winner_model_b  winner_tie
0  3297560222        0.351088        0.314784    0.334128
1  2556155375        0.056825        0.769490    0.173685
2  1793939629        0.306460        0.338921    0.354619
3  2562993561        0.220623        0.448908    0.330469
4  2111345399        0.380912        0.290334    0.328753
